# Editing-cycle simulation — full pipeline

Replaces the human reviewer with an AI judge:

1. **preedit** — read existing robot predictions, parse top-1 *(no GPU)*
2. **judge** — stronger model reviews from image *(Azure API, no GPU)*
3. **postedit** — robot re-diagnoses given feedback *(GPU)*
4. **eval** — pre/judge/post accuracy vs MIDAS ground truth (y16)

Run on a **GPU node** (phase 3 needs it). All outputs go to `prelim_simedit/results_local/`.

In [1]:
import os, sys

PROJECT_ROOT = "/scratch/jq2uw/derm_vlms"
SIMEDIT = os.path.join(PROJECT_ROOT, "prelim_simedit")
if SIMEDIT not in sys.path:
    sys.path.insert(0, SIMEDIT)

from utils.pipeline import run_preedit, run_judge, run_postedit, run_eval
from utils.io import set_output_dir

# --- Configuration ---
ROBOT = "medgemma"         # candidates: medgemma, dermato_llama
JUDGE = "claude_opus48"            # judges: gpt53, gpt54, claude_opus48, claude_opus46, claude_sonnet46, claude_fable
SEED = 42                  # for reproducibility (used wherever randomness is needed)

# Filtering (pick ONE approach):
N = 5                      # first N cases (set None for ALL)
CASE_IDS = None            # OR explicit list: ["1_combined", "5_combined", "10_combined"]

# Notebook test outputs go to results_local/test/ (can overwrite freely).
# Set to None to write to results_local/<robot>/ (official runs).
set_output_dir(os.path.join(SIMEDIT, "results_local", "test"))

In [2]:
# --- Phase 1: preedit (no GPU) ---
# Reads existing predictions CSV, parses top-1 diagnosis.
# Takes first N cases (sorted by case_id). Set CASE_IDS to override.

df_preedit = run_preedit(ROBOT, n=N, seed=SEED, case_ids=CASE_IDS)
df_preedit[["case_id", "gt_y16", "preedit_dx"]]

build_inputs(medgemma): 5 combined cases
[preedit/medgemma] 5 cases -> /scratch/jq2uw/derm_vlms/prelim_simedit/results_local/test/01_preedit.csv


,case_id,gt_y16,preedit_dx
0,1_combined,Squamous Cell Carcinoma In Situ,Basal Cell Carcinoma
1,2_combined,Melanocytic Nevus,Basal Cell Carcinoma
2,6_combined,Squamous Cell Carcinoma,Basal Cell Carcinoma
3,8_combined,Other,Actinic Keratosis
4,9_combined,Seborrheic Keratosis,Basal Cell Carcinoma


In [3]:
# --- Phase 2: judge (Azure API, no GPU) ---
# GPT-5.3 sees the image + robot's top-1 diagnosis.
# Resumable: skips case_ids already in the judge CSV.

df_judge = run_judge(robot_name=ROBOT, judge=JUDGE)
df_judge[["case_id", "preedit_dx", "judge_verdict", "judge_correct_dx", "judge_reasoning"]]

[judge/medgemma/claude_opus48] 5/5 pending


100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:26<00:00,  5.27s/it]

[judge/medgemma/claude_opus48] -> /scratch/jq2uw/derm_vlms/prelim_simedit/results_local/test/02_judge__claude_opus48.csv


,case_id,preedit_dx,judge_verdict,judge_correct_dx,judge_reasoning
0,1_combined,Basal Cell Carcinoma,correct,Basal Cell Carcinoma,"On a sun-damaged background, the dermoscopy sh..."
1,2_combined,Basal Cell Carcinoma,incorrect,Seborrheic keratosis / solar lentigo,"Dermoscopy shows a small, uniformly light-brow..."
2,6_combined,Basal Cell Carcinoma,incorrect,Actinic keratosis (with possible squamous cell...,The lesion sits on a background of severely su...
3,8_combined,Actinic Keratosis,correct,Actinic Keratosis,On sun-damaged dorsal hand skin of an elderly ...
4,9_combined,Basal Cell Carcinoma,incorrect,Squamous cell carcinoma / actinic keratosis,"On a sun-damaged scalp of an elderly patient, ..."


In [4]:
# --- Phase 3: postedit (GPU) ---
# Robot re-diagnoses given the judge's feedback.
# Resumable: skips case_ids already in the postedit CSV.

df_postedit = run_postedit(ROBOT, judge_name=JUDGE)
df_postedit[["case_id", "gt_y16", "preedit_dx", "judge_verdict", "judge_correct_dx", "postedit_dx"]]

[postedit/medgemma/claude_opus48] 5/5 pending


/home/jq2uw/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.37s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Total params: 4,300,079,472


100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:10<00:00,  2.10s/it]

[postedit/medgemma/claude_opus48] -> /scratch/jq2uw/derm_vlms/prelim_simedit/results_local/test/03_postedit__claude_opus48.csv


,case_id,gt_y16,preedit_dx,judge_verdict,judge_correct_dx,postedit_dx
0,1_combined,Squamous Cell Carcinoma In Situ,Basal Cell Carcinoma,correct,Basal Cell Carcinoma,Basal Cell Carcinoma
1,2_combined,Melanocytic Nevus,Basal Cell Carcinoma,incorrect,Seborrheic keratosis / solar lentigo,Seborrheic keratosis / solar lentigo
2,6_combined,Squamous Cell Carcinoma,Basal Cell Carcinoma,incorrect,Actinic keratosis (with possible squamous cell...,Actinic keratosis (with possible squamous cell...
3,8_combined,Other,Actinic Keratosis,correct,Actinic Keratosis,Actinic Keratosis
4,9_combined,Seborrheic Keratosis,Basal Cell Carcinoma,incorrect,Squamous cell carcinoma / actinic keratosis,Squamous cell carcinoma / actinic keratosis


In [5]:
# --- Phase 4: eval (no GPU) ---
# Score all phases against ground truth (y16).

scored, summary = run_eval(robot_name=ROBOT, judge_name=JUDGE)
summary

[eval/medgemma/claude_opus48] -> /scratch/jq2uw/derm_vlms/prelim_simedit/results_local/test/scored__claude_opus48.csv
   robot         judge  n  preedit_acc  postedit_acc  delta_acc  n_improved  n_regressed  judge_dx_acc  judge_verdict_agreement
medgemma claude_opus48  5          0.0           0.2        0.2           1            0           0.2                      0.6


,robot,judge,n,preedit_acc,postedit_acc,delta_acc,n_improved,n_regressed,judge_dx_acc,judge_verdict_agreement
0,medgemma,claude_opus48,5,0.0,0.2,0.2,1,0,0.2,0.6


In [6]:
# --- Per-case details ---
scored[["case_id", "gt_y16", "preedit_dx", "preedit_y16", "preedit_correct",
        "judge_verdict", "judge_correct_dx", "judge_dx_y16", "judge_dx_correct",
        "postedit_dx", "postedit_y16", "postedit_correct"]]

,case_id,gt_y16,preedit_dx,preedit_y16,preedit_correct,judge_verdict,judge_correct_dx,judge_dx_y16,judge_dx_correct,postedit_dx,postedit_y16,postedit_correct
0,1_combined,Squamous Cell Carcinoma In Situ,Basal Cell Carcinoma,Basal Cell Carcinoma,False,correct,Basal Cell Carcinoma,Basal Cell Carcinoma,False,Basal Cell Carcinoma,Basal Cell Carcinoma,False
1,2_combined,Melanocytic Nevus,Basal Cell Carcinoma,Basal Cell Carcinoma,False,incorrect,Seborrheic keratosis / solar lentigo,Seborrheic Keratosis,False,Seborrheic keratosis / solar lentigo,Seborrheic Keratosis,False
2,6_combined,Squamous Cell Carcinoma,Basal Cell Carcinoma,Basal Cell Carcinoma,False,incorrect,Actinic keratosis (with possible squamous cell...,Squamous Cell Carcinoma,True,Actinic keratosis (with possible squamous cell...,Squamous Cell Carcinoma,True
3,8_combined,Other,Actinic Keratosis,Actinic Keratosis,False,correct,Actinic Keratosis,Actinic Keratosis,False,Actinic Keratosis,Actinic Keratosis,False
4,9_combined,Seborrheic Keratosis,Basal Cell Carcinoma,Basal Cell Carcinoma,False,incorrect,Squamous cell carcinoma / actinic keratosis,Squamous Cell Carcinoma,False,Squamous cell carcinoma / actinic keratosis,Squamous Cell Carcinoma,False
